## Data Drift Detection with TrustyAI

Data drift occurs when the distribution of live inference data diverges from the training data, potentially degrading model performance. This notebook demonstrates how to configure TrustyAI to monitor a deployed fraud detection model for data drift.

We will:
1. Upload reference (training) data to TrustyAI
2. Register Kolmogorov-Smirnov (KS) drift monitoring
3. Send in-distribution inference data and confirm no drift is detected
4. Send out-of-distribution inference data and observe drift in the OpenShift Console

### Prerequisites

- Fraud detection ONNX model deployed via KServe in OpenShift AI
- TrustyAI service deployed in the same namespace (`TrustyAIService` CR)
- Pre-computed data files in `data/` (generated by `prepare_data.py`)

### Imports

In [ ]:
from json import load
from time import sleep

from requests import get, post

### Configuration

Set the model name to match the `InferenceService` name of your deployed model (including any suffix). The TrustyAI URL uses the namespace-local service name. Replace the cluster domain and authentication token with your values.

In [ ]:
model_name = 'fraud-detection-r7wdldmu'
namespace = 'fraud-detection'
cluster_domain = 'apps.ocp.wx778.sandbox3406.opentlc.com'

trustyai_service_url = 'http://trustyai-service:8080'
model_inference_url = f'https://{model_name}-{namespace}.{cluster_domain}/v2/models/{model_name}/infer'

authentication_token = 'YOUR_TOKEN'

In [ ]:
def get_from_trustyai(resource_path):
    url = f'{trustyai_service_url}/{resource_path}'
    headers = {'Authorization': f'Bearer {authentication_token}'}
    response = get(url, headers=headers)
    print(response.text)
    return response


def post_to_trustyai(resource_path, payload):
    url = f'{trustyai_service_url}/{resource_path}'
    headers = {'Authorization': f'Bearer {authentication_token}'}
    response = post(url, headers=headers, json=payload)
    print(response.text)
    return response


def send_inference(payload):
    headers = {'Authorization': f'Bearer {authentication_token}'}
    return post(model_inference_url, headers=headers, json=payload)

### Upload Reference Data to TrustyAI

Upload 1000 preprocessed training samples as the reference distribution. We first run inference on the reference data to capture the model's actual outputs, then upload both inputs and outputs to TrustyAI. Using real model outputs ensures the output distribution in the reference matches what the model produces during live inference.

In [ ]:
with open('data/reference_data.json') as f:
    reference_data = load(f)

all_samples = reference_data['inputs'][0]['data']
all_outputs = []
batch_size = 50

print('Getting model outputs for reference data...')
for i in range(0, len(all_samples), batch_size):
    batch = all_samples[i:i + batch_size]
    payload = {
        'inputs': [{
            'name': 'dense_input',
            'shape': [len(batch), 30],
            'datatype': 'FP32',
            'data': batch,
        }]
    }
    response = send_inference(payload)
    batch_outputs = response.json()['outputs'][0]['data']
    all_outputs.extend(batch_outputs)
    sleep(0.5)

print(f'Collected {len(all_outputs)} output predictions.')

upload_payload = {
    'model_name': model_name,
    'data_tag': 'TRAINING',
    'request': {'inputs': reference_data['inputs']},
    'response': {
        'model_name': model_name,
        'model_version': '1',
        'outputs': [{
            'name': 'dense_2',
            'shape': [len(all_samples), 2],
            'datatype': 'FP32',
            'data': all_outputs,
        }],
    },
}

post_to_trustyai('data/upload', payload=upload_payload)

In [ ]:
get_from_trustyai('info')

### Label Data Fields

Map the internal tensor index names to human-readable feature names. The input features are `scaled_amount`, `scaled_time`, and the PCA components `V1`-`V28`.

**Note:** The output tensor name (used in `outputMapping`) depends on the ONNX model's export. Check the `GET /info` response above for the actual output name and adjust if needed.

We also configure drift monitoring to track only input features, since the reference outputs are synthetic and don't match the model's actual output distribution.

In [ ]:
feature_names = ['scaled_amount', 'scaled_time'] + [f'V{i}' for i in range(1, 29)]
input_mapping = {f'dense_input-{i}': name for i, name in enumerate(feature_names)}

info = get_from_trustyai('info')

if f'dense_input-0' in info.text:
    name_mapping = {
        'modelId': model_name,
        'inputMapping': input_mapping,
        'outputMapping': {
            'dense_2-0': 'No Fraud Probability',
            'dense_2-1': 'Fraud Probability',
        },
    }
    post_to_trustyai('info/names', payload=name_mapping)
else:
    print('Feature names already mapped, skipping.')

### Register Drift Monitoring

Register a recurring Kolmogorov-Smirnov (KS) drift test. TrustyAI will compare all inference data against the `TRAINING` reference and emit a Prometheus metric (`trustyai_kstest`) per feature.

The KS test is non-parametric — it compares entire distributions without assuming normality, making it robust for the PCA-derived features in this model. It produces p-values: a value near 1.0 means the data likely comes from the same distribution as the training data, while a value below 0.05 indicates statistically significant drift.

In [ ]:
drift_request = {
    'modelId': model_name,
    'referenceTag': 'TRAINING',
}

post_to_trustyai('metrics/drift/kstest/request', drift_request)

In [ ]:
get_from_trustyai('metrics/all/requests')

### Send In-Distribution Inference Data

Send 500 samples drawn from the same distribution as the training data. TrustyAI captures these automatically via the KServe inference logger. After processing, the KS test p-values should remain near 1.0 for all features.

In [ ]:
with open('data/live_data_normal.json') as f:
    normal_data = load(f)

all_samples = normal_data['inputs'][0]['data']
batch_size = 50
num_batches = len(all_samples) // batch_size

for i in range(0, len(all_samples), batch_size):
    batch = all_samples[i:i + batch_size]
    payload = {
        'inputs': [{
            'name': 'dense_input',
            'shape': [len(batch), 30],
            'datatype': 'FP32',
            'data': batch,
        }]
    }
    response = send_inference(payload)
    print(f'Batch {i // batch_size + 1}/{num_batches}: status {response.status_code}')
    sleep(1)

print('\nIn-distribution data sent.')

### Observe: No Drift Detected

Open the OpenShift Console and navigate to **Observe → Metrics**. Enter `trustyai_kstest` in the expression field and set the time range to 5 minutes.

All features should show p-values near **1.0**, indicating no statistically significant drift between the inference data and the training reference.

### Send Out-of-Distribution Inference Data

Now send 1000 samples where four features have been shifted by 3 standard deviations from their training distribution:

| Feature | Shift | Scenario |
|---------|-------|----------|
| `scaled_amount` | +3 stdev | Transaction amount inflation |
| `V1` | +3 stdev | Shifted spending pattern |
| `V4` | +3 stdev | Shifted spending pattern |
| `V14` | -3 stdev | Shifted spending pattern |

The remaining 26 features are unchanged. After TrustyAI processes this data, drift should be clearly visible for the shifted features.

In [ ]:
with open('data/live_data_drift.json') as f:
    drift_data = load(f)

all_samples = drift_data['inputs'][0]['data']
num_batches = len(all_samples) // batch_size

for i in range(0, len(all_samples), batch_size):
    batch = all_samples[i:i + batch_size]
    payload = {
        'inputs': [{
            'name': 'dense_input',
            'shape': [len(batch), 30],
            'datatype': 'FP32',
            'data': batch,
        }]
    }
    response = send_inference(payload)
    print(f'Batch {i // batch_size + 1}/{num_batches}: status {response.status_code}')
    sleep(1)

print('\nOut-of-distribution data sent.')

### Observe: Drift Detected

Return to the OpenShift Console (**Observe → Metrics**) and refresh the `trustyai_kstest` query.

Expected results:
- **`scaled_amount`, `V1`, `V4`, `V14`**: p-values drop to near **0.0** — statistically significant drift
- **All other features**: p-values remain near **1.0** — no drift

This demonstrates that TrustyAI correctly identifies *which* features have drifted and which remain in-distribution.

### Summary

This demo showed how TrustyAI monitors a deployed fraud detection model for data drift using the Kolmogorov-Smirnov statistical test. By comparing live inference data against a training reference, TrustyAI produces per-feature p-values exposed as Prometheus metrics, making drift immediately visible in the OpenShift Console.

In a production environment, you could define Prometheus alerting rules against `trustyai_kstest` to trigger automated notifications or model retraining workflows when drift is detected.

### Reset for Re-Demo

To clear TrustyAI's stored data and re-run the demo from the beginning, run the following in a terminal:

```bash
oc exec -n fraud-detection deployment/trustyai-service -c trustyai-service -- sh -c 'rm -f /inputs/fraud-detection-*'
oc delete pod -n fraud-detection -l app=trustyai-service
```

Wait for the pod to restart (~30 seconds), then re-run this notebook from the **Upload Reference Data** cell.